In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:

# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
# from tpch import load_lineitem, q06
import pandas as pd
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15" #"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:
# %%time
# ### cell 1 ###

# # Filter and compute total entirely on GPU
# date_filter = (lineitem["L_SHIPDATE"].dt.year == 1996)

# discount_filter = lineitem["L_DISCOUNT"].between(0.08, 0.1)
# quantity_filter = lineitem["L_QUANTITY"] < 24

# mask = date_filter & discount_filter & quantity_filter

# total = (lineitem["L_EXTENDEDPRICE"] * lineitem["L_DISCOUNT"])[mask].sum()

In [6]:
%%time
#original

date1 = pd.Timestamp("1996-01-01")
date2 = pd.Timestamp("1997-01-01")
lineitem_filtered = lineitem.loc[
    :, ["L_QUANTITY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_SHIPDATE"]
]
sel = (
    (lineitem_filtered.L_SHIPDATE >= date1)
    & (lineitem_filtered.L_SHIPDATE < date2)
    & (lineitem_filtered.L_DISCOUNT >= 0.08)
    & (lineitem_filtered.L_DISCOUNT <= 0.1)
    & (lineitem_filtered.L_QUANTITY < 24)
)
flineitem = lineitem_filtered[sel]
total = (flineitem.L_EXTENDEDPRICE * flineitem.L_DISCOUNT).sum()



CPU times: user 1.81 s, sys: 4.86 s, total: 6.67 s
Wall time: 6.77 s


In [7]:
### cell 2 ###

total = total + 1